<a href="https://colab.research.google.com/github/Mojtaba-Choopani/LLM/blob/exercises/E%201-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1.1. Effect of `max_new_tokens=10`

What difference do you observe in the output when running the code with `max_new_tokens=10`?

In [ ]:
!pip install datasets

In [2]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig

In [3]:
huggingface_dataset_name = "knkarthick/dialogsum"

dataset = load_dataset(huggingface_dataset_name)

In [ ]:
model_name='google/flan-t5-base'

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

In [6]:
example_indices = [40, 200]
dash_line = '-'.join('' for x in range(100))

for i, index in enumerate(example_indices):
    dialogue = dataset['test'][index]['dialogue']
    summary = dataset['test'][index]['summary']

    inputs = tokenizer(dialogue, return_tensors='pt')
    output1 = tokenizer.decode(
        model.generate(
            inputs["input_ids"],
            max_new_tokens=50,
        )[0],
        skip_special_tokens=True
    )
    output2 = tokenizer.decode(
        model.generate(
            inputs["input_ids"],
            max_new_tokens=10,
        )[0],
        skip_special_tokens=True
    )

    print(dash_line)
    print('Example ', i + 1)

    print(f'MODEL GENERATION - WITH max_new_tokens=50:\n{output1}\n')
    print(f'MODEL GENERATION - WITH max_new_tokens=10:\n{output2}\n')

---------------------------------------------------------------------------------------------------
Example  1
MODEL GENERATION - WITH max_new_tokens=50:
Person1: It's ten to nine.

MODEL GENERATION - WITH max_new_tokens=10:
Person1: It's ten to nine

---------------------------------------------------------------------------------------------------
Example  2
MODEL GENERATION - WITH max_new_tokens=50:
#Person1#: I'm thinking of upgrading my computer.

MODEL GENERATION - WITH max_new_tokens=10:
#Person1#: I'm thinking



### 1.1. Difference Between `max_new_tokens=50` and `max_new_tokens=10`

`max_new_tokens` limits the number of new tokens generated by the model.  
With `max_new_tokens=10`, the output is shorter and may be incomplete.  
With `max_new_tokens=50`, the model can generate a longer and more complete response.

### 1.2 Effect of `do_sample=True` and `temperature`

After enabling `do_sample=True` and changing the value of `temperature`, what changes do you observe in the generated text?

In [11]:
def make_prompt(example_indices_full, example_index_to_summarize):
    prompt = ''
    for index in example_indices_full:
        dialogue = dataset['test'][index]['dialogue']
        summary = dataset['test'][index]['summary']

        # The stop sequence '{summary}\n\n\n' is important for FLAN-T5. Other models may have their own preferred stop sequence.
        prompt += f"""
Dialogue:

{dialogue}

What was going on?
{summary}


"""

    dialogue = dataset['test'][example_index_to_summarize]['dialogue']

    prompt += f"""
Dialogue:

{dialogue}

What was going on?
"""

    return prompt

In [12]:
example_indices_full = [40, 80, 120]
example_index_to_summarize = 200

few_shot_prompt = make_prompt(example_indices_full, example_index_to_summarize)

print(few_shot_prompt)


Dialogue:

#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.

What was going on?
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.



Dialogue:

#Person1#: May, do you mind helping me prepare for the picnic?
#Person2#: Sure. Have you checked the weather report?
#Person1#: Yes. It says it will be sunny all day. No sign of rain at all. This is your father's favorite sausage. Sandwiches for you and Daniel.
#Person2#: No, thanks Mom. I'd like some toast and chicken wings.
#Person1#: Okay. Please take some fruit salad and crackers for me.
#Person2#: Done. Oh, don't forget to take napkins disposable plates, cups and picnic blanket.
#Person1#: All set. 

In [15]:
summary = dataset['test'][example_index_to_summarize]['summary']

generation_config1 = GenerationConfig(max_new_tokens=50)
generation_config2 = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=0.1)
generation_config3 = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=0.5)
generation_config4 = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=1.0)

inputs = tokenizer(few_shot_prompt, return_tensors='pt')
output1 = tokenizer.decode(
    model.generate(
        inputs["input_ids"],
        generation_config1,
    )[0],
    skip_special_tokens=True
)
output2 = tokenizer.decode(
    model.generate(
        inputs["input_ids"],
        generation_config2,
    )[0],
    skip_special_tokens=True
)
output3 = tokenizer.decode(
    model.generate(
        inputs["input_ids"],
        generation_config3,
    )[0],
    skip_special_tokens=True
)
output4 = tokenizer.decode(
    model.generate(
        inputs["input_ids"],
        generation_config4,
    )[0],
    skip_special_tokens=True
)



print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - FEW SHOT-max_new_tokens=50 :\n{output1}')
print(dash_line)
print(f'MODEL GENERATION - FEW SHOT-max_new_tokens=50, do_sample=True, temperature=0.1 :\n{output2}')
print(dash_line)
print(f'MODEL GENERATION - FEW SHOT-max_new_tokens=50, do_sample=True, temperature=0.5 :\n{output3}')
print(dash_line)
print(f'MODEL GENERATION - FEW SHOT-max_new_tokens=50, do_sample=True, temperature=1.0 :\n{output4}')


---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.

---------------------------------------------------------------------------------------------------
MODEL GENERATION - FEW SHOT-max_new_tokens=50 :
#Person1 wants to upgrade his system. #Person2 wants to add a painting program to his software. #Person1 wants to upgrade his hardware.
---------------------------------------------------------------------------------------------------
MODEL GENERATION - FEW SHOT-max_new_tokens=50, do_sample=True, temperature=0.1 :
#Person1 recommends upgrading the system and hardware.
---------------------------------------------------------------------------------------------------
MODEL GENERATION - FEW SHOT-max_new_tokens=50, do_sample=True, temperature=0.5 :
#Person2 suggests the following things: painting program, a better computer, a better 

The results show that using do_sample=True with a low temperature such as 0.1 produces a more concise and controlled summary that is closer to the human baseline. As the temperature increases, the output becomes more diverse, but it also becomes less accurate and may include irrelevant or less faithful information.